# TENOR-SAXS — variable-resolution scattering demo

A hands-on, interactive demonstration of **TENOR-SAXS** (Technique for ENsemble Observation by Resolution variation in SAXS), from *"Variable-Resolution Scattering Reveals Ensemble Properties"* (Steinitz & Beck).

The method recovers a nanoparticle ensemble's polydispersity — the scattering-weighted relative variance of the radius of gyration, $V = \mathrm{Var}_w(R_g)/\langle R_g\rangle_w^2$ — directly from a **single 2D SAXS image**, by digitally smearing it with two different anisotropic point-spread functions and fitting the log-ratio of the two smeared images, without needing to know the instrument's own native resolution function.

Code: **https://github.com/roybeckbarkai/tenor-saxs** (clean-room Python port, validated against the paper's MATLAB reference implementation — see `the internal validation notes` in the repo for every difference found and how it was resolved).

This notebook:
1. Installs the package directly from GitHub.
2. Provides an **interactive control panel** (dropdowns + input fields) to simulate a polydisperse ensemble and run the TENOR-SAXS protocol on it, with no code editing required.
3. Reports both the raw **apparent** (Guinier) $R_g$ and the **polydispersity-corrected** mean $R_g$ (derived analytically from the recovered $V$ — see the note in cell below).
4. Sweeps over a range of true $V$ values to show recovery accuracy.
5. Runs a small noise-sensitivity benchmark, reproducing the paper's own validation approach.

Runtime: a standard (CPU-only) Colab runtime is sufficient — nothing here needs a GPU.

In [ ]:
# --no-deps on the git install: Colab already ships numpy/scipy/pandas/matplotlib
# that satisfy our requirements, and force-reinstalling THOSE too (pulling a
# different numpy build than the one Colab's other preinstalled packages were
# built against) causes a numpy/BLAS ABI mismatch (AttributeError mentioning
# `_blas_supports_fpe` or similar on import). --force-reinstall --no-deps only
# touches the tiny tenor-saxs package itself, so Colab's own stack stays intact.
# If you still hit an import error -- or a TypeError/AttributeError about a missing
# or unexpected argument on a function that used to work -- it means an OLDER build of
# numpy or tenor_saxs is still loaded in this process from a previous run in this
# session (re-running this install cell alone does NOT reload code already imported
# into a live kernel). Use Runtime > Restart session (not just re-running cells) and
# run all cells again from the top.
%pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/roybeckbarkai/tenor-saxs.git
%pip install -q ipywidgets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from tenor_saxs import psf, simulation, protocol, benchmark, plotting, formfactors, calibration, distributions, display_names
from tenor_saxs.bimodal_skewed import BimodalSkewedInfeasible

np.set_printoptions(precision=4, suppress=True)


def signed_sqrt(v):
    """sqrt(V) for V>=0, sign(V)*sqrt(|V|) for V<0 -- keeps negative outliers
    visible on a sqrt(V) axis instead of being dropped (matches the paper's
    V^(1/2) convention; used throughout this notebook for all V axes/values)."""
    v = np.asarray(v, dtype=float)
    return np.sign(v) * np.sqrt(np.abs(v))


print("tenor_saxs imported OK")


## 1. Interactive control panel

Pick a form factor, distribution shape, mean radius of gyration $R_0$, true polydispersity $V$, and noise level, then click **Run** to simulate a 2D SAXS pattern and recover $V$/$R_g$ from it with TENOR-SAXS.

**Apparent vs. corrected $R_g$:** the direct Guinier-fit $R_g$ from a polydisperse ensemble is always biased *upward* — $R_{g,\mathrm{apparent}} = R_0\sqrt{1+V}$ (paper Eq. 5). Once TENOR-SAXS recovers $V$, that bias can be undone analytically: $R_{0,\mathrm{corrected}} = R_{g,\mathrm{apparent}}/\sqrt{1+V_{\mathrm{recovered}}}$ — the same first-order-in-$V$ relationship used throughout the method, requiring no assumption about which distribution shape actually produced the data (only $R_0$ and $V$ themselves enter, to this order). Both values are reported below.

In [ ]:
#@title ▶ TENOR-SAXS interactive control panel  (double-click, or the ⋮ menu, to view/edit the code) { display-mode: "form" }
FORM_FACTORS = {
    "Gaussian chain (φ''=1/18)": "gaussian_chain",
    "Solid sphere (φ''=-1/63)": "solid_sphere",
    "Spherical shell (φ''=-1/45)": "spherical_shell",
    "Thin rod (φ''=11/225)": "thin_rod",
}
DISTRIBUTIONS = [
    ("normal", "normal"),
    ("lognormal", "lognormal"),
    ("schulz", "schulz"),
    ("boltzmann", "boltzmann"),
    ("triangular", "triangular"),
    ("uniform", "uniform"),
    ("Bimodal, skewed — exact 2-point, uses the γ slider below", "bimodal_skewed"),
]
WEIGHT_MODES = [("intensity — default: matches the paper and MATLAB, and is the statistically optimal choice under photon-counting noise", "intensity"),
                ("sqrt(intensity) — an earlier draft's alternative convention", "sqrt_intensity")]
YG2_YM_ROUTES = [("analytic (default)", "analytic"),
                 ("simulated calibration — recommended for Y_G2/Y_M", "calibration")]

# Smearing-filter (pxn) presets matching the paper's Table 1 "Smearing PSFs
# (pixels of 4σ kernel)" column -- these are the two ANALYSIS-time digital
# Gaussian filters (distinct from psf0 below, the SIMULATION-time native PSF).
PXN_PRESETS = {
    "Baseline [125×123, 87×85] (paper Table 1)": np.array([87, 85, 125, 123]),
    "Noise benchmark [91×81, 117×127] (paper Table 1)": np.array([91, 81, 117, 127]),
    "Custom": None,
}

# [guardrail] compact-support distributions under high scattering-weight power have a hard
# ceiling on achievable scattering-weighted V (confirmed empirically: triangular/uniform at
# weight_power=4 (spherical shell) or 6 (solid sphere) peak around V~0.02-0.05, far below the
# calibration curve's default V-node grid) -- building a calibration curve for these always
# fails with "only N/33 calibration nodes succeeded". Not a bug to work around per-case; the
# analytic route is used instead and a warning is shown if a user picks this combination anyway.
_CALIBRATION_UNSUPPORTED = {
    ("triangular", "solid_sphere"), ("triangular", "spherical_shell"),
    ("uniform", "solid_sphere"), ("uniform", "spherical_shell"),
}

style = {"description_width": "140px"}
layout = widgets.Layout(width="420px")

# Long explanatory text lives in each control's hover tooltip (`tooltip=`) rather than as
# always-visible paragraphs, to keep the panel itself compact. Hover a label to read it.
form_factor_dd = widgets.Dropdown(options=list(FORM_FACTORS.keys()), value="Gaussian chain (φ''=1/18)", description="Form factor:", style=style, layout=layout)
distribution_dd = widgets.Dropdown(
    options=DISTRIBUTIONS, value="normal", description="Distribution:", style=style, layout=layout,
    tooltip=("Bimodal, skewed is an exact two-point distribution with a prescribed scattering-"
             "weighted (mean, V, gamma) -- set its skewness with the gamma slider; not every "
             "(V, gamma) combination is realizable under a minimum-radius floor (0.5x mean here, "
             "matching the reference benchmark), in which case you'll see a message explaining "
             "why instead of a crash. gamma=0 requires V>0 (skewness is undefined for a point "
             "mass). A sweep of this panel's parameter space also found that large positive "
             "gamma combined with V>=0.2 often yields no usable observable (the two support "
             "points become too asymmetric for a good Guinier fit) -- this shows up as all-red "
             "points below, not a crash, and is an inherent limitation of the extreme two-point "
             "construction at that corner, not a bug."))
r0_input = widgets.FloatText(value=3.0, description="R0 (nm):", style=style, layout=layout)
v_slider = widgets.FloatSlider(value=0.08, min=0.0, max=0.3, step=0.01, description="True V:", style=style, layout=layout, readout_format=".3f")
n_radii_input = widgets.IntText(value=25, description="N radii:", style=style, layout=layout)
gamma_slider = widgets.FloatSlider(value=0.0, min=-1.0, max=2.0, step=0.1, description="Bimodal γ:", style=style, layout=layout, readout_format=".2f", disabled=True)
noise_mode_dd = widgets.Dropdown(options=["Noise-free", "Photon-counting noise"], value="Photon-counting noise", description="Noise mode:", style=style, layout=layout)
# Range extended to 1e8: Y_M/J_M (pure functions of the m2/m1 ratio) need roughly
# 1e4x more photons than Y_G/J_G to approach their noise-free value at the baseline
# smearing quartet -- see peak_photons_slider's tooltip. Runs above ~1e7 take noticeably longer.
peak_photons_slider = widgets.FloatLogSlider(
    value=1e4, base=10, min=1, max=8, step=0.1, description="Peak photons (per pixel):", style=style, layout=layout,
    tooltip=("Y_M/J_M convergence: both are pure functions of the m2/m1 ratio, and at the "
             "'Baseline' smearing quartet that ratio sits close to the fit's noise floor -- "
             "Y_M/J_M need roughly 10^4x more photons than Y_G/J_G to approach their noise-free "
             "value (confirmed by direct simulation, not a bug). Try the 'Noise benchmark' "
             "smearing quartet (more anisotropic, better-conditioned) and/or push this toward "
             "10^7-10^8 to see them converge."))
seed_input = widgets.IntText(value=0, description="Random seed:", style=style, layout=layout)
keep_seed_checkbox = widgets.Checkbox(value=False, description="Keep seed constant across runs", indent=False, layout=widgets.Layout(width="400px"))

weight_mode_dd = widgets.Dropdown(options=WEIGHT_MODES, value="intensity", description="Fit weighting:", style=style, layout=layout)
use_g3_checkbox = widgets.Checkbox(value=False, description="Fit g3/m3 (3rd order)", indent=False, layout=widgets.Layout(width="400px"))
# [guardrail] a systematic sweep of this panel's own parameter space (~1800 combinations)
# found two reliable crash conditions here: qrg_max below ~0.1-0.2 (the Guinier window becomes
# too narrow to fit, raising "expected square matrix" or giving an all-unusable result), and
# deadpix_factor above ~3 at this panel's default geometry (the border-exclusion margin eats
# the whole window, raising "not enough pixels" from mg_extract). Both are now caught below with
# a clear message instead of a raw traceback, but staying inside roughly qrg_max in [0.3, 1.5]
# and deadpix_factor in [0, 2] avoids them entirely.
_window_tooltip = ("Analysis window: upper Guinier edge as q*R0(apparent), and the detector-"
                    "border margin in units of the widest smearing kernel. The window is the "
                    "narrower of the two, so raising the edge alone cannot widen the window past "
                    "whatever the border margin already caps it at. Keep qRg window roughly in "
                    "[0.3, 1.5] and border margin in [0, 2] -- outside that, the window can close "
                    "entirely (see the message below the Run button if it does).")
qrg_max_input = widgets.FloatText(value=protocol.QRG_MAX, step=0.05, description="qRg window:", style=style, layout=layout, tooltip=_window_tooltip)
deadpix_factor_input = widgets.FloatText(value=protocol.DEADPIX_FACTOR, step=0.5, description="Border margin:", style=style, layout=layout, tooltip=_window_tooltip)
yg2_ym_route_dd = widgets.Dropdown(
    options=YG2_YM_ROUTES, value="analytic", description="Y_G2/Y_M route:", style=style, layout=layout,
    tooltip=("Y_G2/Y_M inversion route: their analytic (Table 2) formulas are measurably biased "
             "and, for Y_G2, non-invertible past V~0.27; the simulated calibration route (a curve "
             "built by actually simulating this form factor over a V grid, using the current qRg "
             "window/border margin/smearing quartet) fixes both, at the cost of one extra "
             "~5-10s simulation the first time a given combination of settings is used. Not "
             "available for Bimodal, skewed -- the calibration curve builder assumes the "
             "standard discretize_ensemble() families. Also unsupported for {triangular, "
             "uniform} x {solid sphere, spherical shell}: these compact-support distributions "
             "have a hard ceiling on achievable scattering-weighted V well below the calibration "
             "grid's range, so curve-fitting always fails for that combination -- the analytic "
             "route is used instead automatically, with a warning."))

# --- native simulation-time PSF (psf0): anisotropic, user-set, capped at 100x100 ---
_psf0_tooltip = ("Native simulation PSF (psf0): the instrument's own point-spread function, used "
                  "only to simulate a realistic image; TENOR-SAXS's analysis never needs to know "
                  "this. Anisotropic, each axis capped at 100 pixels.")
psf0_x_input = widgets.IntText(value=3, description="PSF0 x (pix):", style=style, layout=layout, tooltip=_psf0_tooltip)
psf0_y_input = widgets.IntText(value=15, description="PSF0 y (pix):", style=style, layout=layout, tooltip=_psf0_tooltip)

# --- analysis-time smearing-filter quartet (pxn): paper-Table-1 presets, or custom ---
pxn_preset_dd = widgets.Dropdown(
    options=list(PXN_PRESETS.keys()), value="Baseline [125×123, 87×85] (paper Table 1)",
    description="Smearing quartet:", style=style, layout=layout,
    tooltip=("Digital smearing filters (pxn): the two analysis-time Gaussian kernels from the "
             "paper's Table 1; this is the parameter that actually drives the method, including "
             "the Y_M/J_M sensitivity (see the Peak photons tooltip)."))
_pxn_custom_labels = ["nx1:", "ny1:", "nx2:", "ny2:"]
_pxn_custom_tooltip = ("Rounded to the nearest odd integer in [1,99]; (nx1,ny1) must also differ "
                        "from (nx2,ny2) as a pair -- both are requirements of the underlying "
                        "Gaussian-kernel construction, not arbitrary limits.")
pxn_custom_inputs = [widgets.IntText(value=v, description=lbl, style={"description_width": "40px"},
                                      layout=widgets.Layout(width="140px"), tooltip=_pxn_custom_tooltip)
                      for v, lbl in zip([87, 85, 125, 123], _pxn_custom_labels)]
pxn_custom_box = widgets.HBox(pxn_custom_inputs, layout=widgets.Layout(display="none"))


def _on_pxn_preset_change(change):
    pxn_custom_box.layout.display = "flex" if change["new"] == "Custom" else "none"


pxn_preset_dd.observe(_on_pxn_preset_change, names="value")


def _current_pxn():
    preset = PXN_PRESETS[pxn_preset_dd.value]
    if preset is not None:
        return preset
    # Custom entries only are capped at 100x100 -- the two named presets above use the
    # paper's own values verbatim even though two of those (117, 127) exceed 100.
    # Also rounded to the nearest odd integer: psf.validate_pxn requires all four
    # entries odd (MG_extract.m:55-66), so an even custom entry would otherwise fail
    # with a "must be positive odd integers" error every time.
    vals = []
    for w in pxn_custom_inputs:
        clamped = min(99, max(1, int(w.value)))
        if clamped % 2 == 0:
            clamped += 1 if clamped < 99 else -1
        w.value = clamped
        vals.append(clamped)
    return np.array(vals)


run_button = widgets.Button(description="Run simulation + analysis", button_style="success", icon="play", layout=widgets.Layout(width="260px", margin="12px 0 0 0"))
output_area = widgets.Output()


def _on_form_factor_change(change):
    shape = FORM_FACTORS[change["new"]]
    wp = formfactors.GUINIER_TABLE[shape].weight_power
    form_factor_dd.tooltip = (
        f"Scattering weight power for this form factor: w={wp:g} (ensemble members weighted "
        f"by R_g^{wp:g}). This is locked to the form factor for physical consistency between "
        f"phi'' and the weighting -- scatter2d silently overrides any other value for these "
        f"four canonical shapes, so it's shown here (hover) rather than offered as a free control."
    )


form_factor_dd.observe(_on_form_factor_change, names="value")

# Layout: three stacked sections (Ensemble / Instrument-PSF / Analysis), not side by side.
# Explanatory text lives in each control's tooltip (hover to read) rather than as visible
# paragraphs, to keep the panel compact.
sim_box = widgets.VBox([widgets.HTML("<b>Ensemble</b>"), form_factor_dd, distribution_dd,
                         r0_input, v_slider, gamma_slider, n_radii_input, noise_mode_dd, peak_photons_slider,
                         seed_input, keep_seed_checkbox])
psf_box = widgets.VBox([widgets.HTML("<b>Instrument / PSF</b>"),
                         psf0_x_input, psf0_y_input,
                         pxn_preset_dd, pxn_custom_box])
analysis_box = widgets.VBox([widgets.HTML("<b>Analysis</b>"), weight_mode_dd, use_g3_checkbox,
                             qrg_max_input, deadpix_factor_input,
                             yg2_ym_route_dd,
                             run_button])
panel = widgets.VBox([sim_box, psf_box, analysis_box])

_calibration_cache = {}  # (phi2, r0, weight_power, distribution, qrg_max, deadpix_factor, pxn, use_g3) -> CalibrationCurve


def _get_calibration_curve(phi2, r0, weight_power, distribution, qrg_max, deadpix_factor, pxn, use_g3):
    key = (phi2, r0, weight_power, distribution, qrg_max, deadpix_factor, tuple(int(v) for v in pxn), use_g3)
    if key not in _calibration_cache:
        _calibration_cache[key] = calibration.build_calibration_curve(
            r0=r0, phi2=phi2, weight_power=weight_power, distribution=distribution,
            qrg_max=qrg_max, deadpix_factor=deadpix_factor, pxn=pxn, use_g3=use_g3, use_r3=use_g3,
        )
    return _calibration_cache[key]


def _on_distribution_change(change):
    dist = change["new"]
    n_radii_input.disabled = (dist == "bimodal_skewed")
    gamma_slider.disabled = (dist != "bimodal_skewed")
    yg2_ym_route_dd.disabled = (dist == "bimodal_skewed")


distribution_dd.observe(_on_distribution_change, names="value")
_on_distribution_change({"new": distribution_dd.value})  # apply initial state
_on_form_factor_change({"new": form_factor_dd.value})    # apply initial state


def run_simulation_and_analysis(_button=None):
    with output_area:
        clear_output(wait=True)
        shape = FORM_FACTORS[form_factor_dd.value]
        phi2 = formfactors.GUINIER_TABLE[shape].phi2
        weight_power = formfactors.GUINIER_TABLE[shape].weight_power
        noise = 0 if noise_mode_dd.value == "Noise-free" else -abs(peak_photons_slider.value)
        used_seed = seed_input.value
        rng = np.random.default_rng(used_seed)
        dist = distribution_dd.value
        psf0 = psf.bartlett2d(min(100, max(1, psf0_x_input.value)), min(100, max(1, psf0_y_input.value)))
        pxn = _current_pxn()
        use_g3 = use_g3_checkbox.value

        try:
            sim = simulation.scatter2d(
                rg=r0_input.value, noise=noise, v_rel=v_slider.value, phi2=phi2,
                det_pix=500, sd_dist=360.0, wavelength=0.1, det_side=3.5,
                psf0=psf0, dist_type=dist,
                n_radii=n_radii_input.value, weight_power=weight_power, rng=rng,
                gamma=gamma_slider.value, bimodal_min_radius=0.5,
            )
        except BimodalSkewedInfeasible as exc:
            print("This (V, γ) combination isn't realizable under the bimodal construction's "
                  "minimum-radius floor:")
            print(f"  {exc}")
            print("Try a smaller |γ| and/or a smaller V.")
            return
        except ValueError as exc:
            if "skewness is undefined" in str(exc):
                print("Bimodal, skewed: skewness is undefined at V=0 (a point mass has no shape). "
                      "Either set γ=0, or raise V above 0.")
            else:
                print(f"Simulation failed to build the ensemble: {exc}")
            return

        # The ACTUAL scattering-weighted V/mean-Rg/RMS-Rg of the ensemble that was just
        # simulated -- not the requested/target value above, which a finite n_radii
        # discretization only approximately hits. Returns (mean, V, rms), in that order.
        realized_mean_rg, realized_v, realized_rms_rg = distributions.scattering_weighted_stats(
            sim.r_used, sim.p_used, weight_power)

        calibration_override = None
        want_calibration = yg2_ym_route_dd.value == "calibration" and dist != "bimodal_skewed"
        if want_calibration and (dist, shape) in _CALIBRATION_UNSUPPORTED:
            print(f"Note: the simulated calibration route isn't available for {dist} x {shape} "
                  "(this compact-support distribution can't reach a high enough scattering-weighted "
                  "V under this form factor's weighting for the calibration grid to fit) -- falling "
                  "back to the analytic route automatically.")
            want_calibration = False
        try:
            if want_calibration:
                print("Building simulated calibration curve (cached after the first run for these settings)...")
                curve = _get_calibration_curve(phi2, r0_input.value, weight_power, dist,
                                                qrg_max_input.value, deadpix_factor_input.value, pxn, use_g3)
                calibration_override = {name: curve.curves[name] for name in ("Yg210", "Ym210")}
            r = protocol.tenor_protocol(
                sim.intensity, sim.qx, sim.qy, phi2=phi2, pxn=pxn,
                weight_mode=weight_mode_dd.value, strategy="inverseVariance",  # fixed: paper's default, not user-exposed
                use_g3=use_g3, use_r3=use_g3,
                qrg_max=qrg_max_input.value, deadpix_factor=deadpix_factor_input.value,
                calibration_override=calibration_override,
            )
        except RuntimeError as exc:
            if "calibration nodes succeeded" in str(exc):
                print("The simulated calibration curve couldn't be built for this form factor/"
                      "distribution/R0 combination (too few V-grid nodes were reachable). Switch "
                      "the Y_G2/Y_M route to 'analytic' for this combination.")
            else:
                print(f"Analysis failed: {exc}")
            return
        except ValueError as exc:
            if "not enough pixels" in str(exc):
                print("Analysis failed: the analysis window closed entirely (border margin too "
                      "large relative to the qRg window and detector geometry).")
                print("Try lowering 'Border margin' and/or raising 'qRg window'.")
            elif "square matrix" in str(exc) or "No valid samples" in str(exc):
                print("Analysis failed: the qRg window is too narrow to fit (too few pixels "
                      "survive it).")
                print("Try raising 'qRg window' (roughly 0.3-1.5 is a safe range).")
            else:
                print(f"Analysis failed: {exc}")
            return

        if dist == "bimodal_skewed":
            print(f"Note: bimodal_skewed targets the SCATTERING-weighted (mean, V, γ) exactly -- "
                  f"requested V = {v_slider.value:.4f}, realized V = {realized_v:.4f}, "
                  f"γ = {gamma_slider.value:.2f}.")
        else:
            print(f"Realized scattering-weighted V = {realized_v:.4f}  (requested {v_slider.value:.4f}; "
                  f"a finite number of discrete radii only approximately hits the target).")

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        im = axes[0].imshow(np.log10(np.clip(sim.intensity, 1e-6, None)),
                             extent=[sim.qx.min(), sim.qx.max(), sim.qy.min(), sim.qy.max()], cmap="viridis")
        axes[0].set_xlabel("$q_x$ (nm$^{-1}$)"); axes[0].set_ylabel("$q_y$ (nm$^{-1}$)")
        axes[0].set_title("Simulated 2D SAXS pattern (log intensity)")
        fig.colorbar(im, ax=axes[0], label="$\\log_{10} I(q)$")

        names = list(r.v_estimates.keys())
        v_vals = np.array([r.v_estimates[n] for n in names], dtype=float)
        v_ses = np.array([r.v_se[n] for n in names], dtype=float)
        sqrt_v = signed_sqrt(v_vals)
        sqrt_lo = signed_sqrt(v_vals - v_ses)
        sqrt_hi = signed_sqrt(v_vals + v_ses)
        colors = ["tab:green" if r.status[n] == "ok" else "tab:red" for n in names]
        axes[1].errorbar(range(len(names)), sqrt_v,
                          yerr=[np.clip(sqrt_v - sqrt_lo, 0, None), np.clip(sqrt_hi - sqrt_v, 0, None)],
                          fmt="o", capsize=4, ecolor="gray")
        for i, c in enumerate(colors):
            axes[1].plot(i, sqrt_v[i], "o", color=c)
        axes[1].axhline(signed_sqrt(realized_v), color="k", linestyle="--", label="realized $\\sqrt{V}$")
        axes[1].axhline(signed_sqrt(r.best_v), color="tab:blue", linestyle=":", label="combined (inverseVariance)")
        axes[1].set_xticks(range(len(names)))
        axes[1].set_xticklabels([display_names.PAPER_LABELS.get(n, n) for n in names], rotation=30)
        axes[1].set_ylabel("$\\sqrt{V}$ estimate")
        axes[1].set_ylim(-0.1, 0.6)
        axes[1].legend(fontsize=8)
        axes[1].set_title("Per-observable $\\sqrt{V}$ (green=ok, red=unusable)")
        fig.tight_layout()
        plt.show()

        if not np.isfinite(r.best_v):
            print("All observables came back unusable for this combination (see the all-red "
                  "plot above) -- this is a real limitation of the method/settings at this point "
                  "in parameter space, not a crash. Try a smaller V, a smaller |γ| (if using "
                  "Bimodal, skewed), or more photons.")
            print()

        y_cols = ["Yg100", "Yg210", "Ym210"]
        col_labels = [display_names.PAPER_LABELS[n] for n in y_cols]
        results_table = pd.DataFrame(
            {label: [
                r.v_estimates[n],
                signed_sqrt(r.v_estimates[n]),
                r.v_estimates[n] - realized_v,
                signed_sqrt(r.v_estimates[n]) - signed_sqrt(realized_v),
                r.rg,
                r.rg_corrected,
            ] for n, label in zip(y_cols, col_labels)},
            index=["V", "sqrt(V)", "V discrepancy (recovered-realized)", "sqrt(V) discrepancy",
                   "Rg (apparent, combined)", "Rg (corrected, combined)"],
        )
        results_table.insert(0, "Realized", [
            realized_v, signed_sqrt(realized_v), np.nan, np.nan, realized_rms_rg, np.nan,
        ])
        print("Results (Rg rows use the single combined estimate, not per-observable -- "
              "tenor_protocol only returns one combined Rg):")
        display(results_table.map(lambda v: "--" if pd.isna(v) else f"{v:.4f}"))

        print()
        print("Per-observable detail (all 6 observables):")
        for n in names:
            print(f"  {display_names.PAPER_LABELS.get(n, n):>16s} ({n:8s}): V={r.v_estimates[n]:.4f}  SE={r.v_se[n]:.4f}  status={r.status[n]!r}")

        if not keep_seed_checkbox.value:
            seed_input.value = used_seed + 1


run_button.on_click(run_simulation_and_analysis)
display(panel, output_area)
run_simulation_and_analysis()  # run once with the defaults so the panel isn't empty


## 2. Recovery accuracy across a range of true $V$

Sweeping the true variance and comparing to what TENOR-SAXS recovers, noise-free — this reproduces (a small version of) the paper's own Fig. 3 validation.


In [ ]:
TRUE_RG = 3.0
FORM_FACTOR_LABEL = "Gaussian chain"
PHI2 = formfactors.GUINIER_TABLE["gaussian_chain"].phi2
WEIGHT_POWER = formfactors.GUINIER_TABLE["gaussian_chain"].weight_power
PSF0 = psf.bartlett2d(3, 15)

v_true_grid = np.linspace(0.0, 0.25, 11)
realized_v_grid = []
v_recovered = {name: [] for name in protocol.OBSERVABLE_NAMES}
rg_apparent_list = []
rg_corrected_list = []
for v_true in v_true_grid:
    s = simulation.scatter2d(
        rg=TRUE_RG, noise=0, v_rel=v_true, phi2=PHI2,
        det_pix=500, sd_dist=360.0, wavelength=0.1, det_side=3.5,
        psf0=PSF0, dist_type="normal", n_radii=25, weight_power=WEIGHT_POWER,
    )
    # Realized (actual), not requested, scattering-weighted V -- the discretizer only
    # approximately hits v_true for a finite n_radii. Returns (mean, V, rms), in that order.
    _mean_rg, realized_v, _rms_rg = distributions.scattering_weighted_stats(s.r_used, s.p_used, WEIGHT_POWER)
    realized_v_grid.append(realized_v)
    r = protocol.tenor_protocol(s.intensity, s.qx, s.qy, phi2=PHI2)
    for name in protocol.OBSERVABLE_NAMES:
        v_recovered[name].append(r.v_estimates[name])
    rg_apparent_list.append(r.rg)
    rg_corrected_list.append(r.rg_corrected)

realized_v_grid = np.array(realized_v_grid)
sqrt_true = signed_sqrt(realized_v_grid)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(sqrt_true, sqrt_true, "k--", label="perfect recovery")
for name in protocol.OBSERVABLE_NAMES:
    axes[0].plot(sqrt_true, signed_sqrt(np.array(v_recovered[name])), "o-", markersize=4,
                 label=display_names.PAPER_LABELS[name])
axes[0].set_xlabel("$\\sqrt{V}$ (realized)"); axes[0].set_ylabel("$\\sqrt{V}$ (recovered)")
axes[0].legend(fontsize=8); axes[0].set_title(f"Noise-free V recovery ({FORM_FACTOR_LABEL}, R0={TRUE_RG:g}nm)")

rg_apparent_disc = (np.array(rg_apparent_list) - TRUE_RG) / TRUE_RG
rg_corrected_disc = (np.array(rg_corrected_list) - TRUE_RG) / TRUE_RG
axes[1].axhline(0.0, color="k", linestyle="--", label="perfect recovery")
axes[1].plot(sqrt_true, rg_apparent_disc, "o-", label="apparent (Guinier)")
axes[1].plot(sqrt_true, rg_corrected_disc, "s-", label="corrected")
axes[1].set_xlabel("$\\sqrt{V}$ (realized)"); axes[1].set_ylabel("$\\Delta R_0/R_0$")
axes[1].legend(); axes[1].set_title("Relative $R_g$ discrepancy vs. true $V$")
fig.tight_layout()
plt.show()


## 3. Noise-sensitivity benchmark

A small-scale version of the paper's Figs. 5-6 noise-sensitivity study: sweep over a few $V$ values and photon-flux levels, with multiple noisy replicates each, and look at how the $\sqrt{V}$-recovery and $R_0$ discrepancies shrink as photon counts increase. This uses `tenor_saxs.benchmark`, which also handles the (nontrivial) job of finding simulation inputs that realize a *scattering-weighted* target variance, since the discretizer only directly controls the *number*-weighted one.

The ensemble/instrument properties below (`TRUE_RG`, `PHI2`, `WEIGHT_POWER`, `PSF0`) are the same variables set in Section 2's cell above — edit that cell and re-run both to change them consistently.

This cell takes roughly 15-30 seconds on a standard Colab CPU runtime (reduce `n_cases`/`n_replicates` for a faster, coarser look).


In [ ]:
import time

config = benchmark.BenchmarkConfig(
    output_root="/content/tenor_benchmark_demo",
    n_replicates=10,
    v_values=np.array([0.01, 0.09, 0.25]),          # a few V values instead of the full 12
    peak_photons=np.array([1e3, 1e4, 1e5]) / 1.68,   # /1.68 -> round numbers on the photon-DENSITY x-axis
    # Ensemble/instrument properties mirror Section 2's cell above (TRUE_RG, PHI2, WEIGHT_POWER,
    # PSF0) -- edit both cells together to keep Sections 2 and 3 consistent.
    rg=TRUE_RG, phi2=PHI2, weight_power=WEIGHT_POWER, psf0=PSF0, distribution="normal",
)

t0 = time.time()
results_df, manifest = benchmark.run_benchmark(config)
# np.isfinite(BestV), NOT Status=="ok" -- the latter only means tenor_protocol
# didn't raise (a much weaker condition: the observable can still come back
# unusable at low photon counts without raising anything).
valid_frac = np.isfinite(results_df["BestV"]).mean()
print(f"Done in {time.time()-t0:.1f}s, {len(results_df)} rows, {valid_frac:.0%} valid")

# The manuscript's current benchmark figure is the (V, R0) pair -- the
# apparent-radius panel was dropped (it has no physical interest: fixed by
# construction, so its own discrepancy is a tautology).
fig, axes = plotting.plot_tenor_benchmark_violins(
    results_df, panels=("V", "R0"), xlabel="Collected photon density (photons/nm$^{2}$)")
fig.suptitle("Noise-sensitivity demo (small-scale: 3 V values x 3 photon levels x 10 replicates)")
plt.show()

print(f"Simulation params used: {FORM_FACTOR_LABEL}, R0={TRUE_RG:g}nm, phi''={PHI2:.4f}, "
      f"weight_power={WEIGHT_POWER:g}, PSF0 shape={PSF0.shape}, distribution={config.distribution!r}")

summary = (
    results_df
    .assign(V_disc=lambda d: d["BestV"] - d["True_V"],
            R_disc=lambda d: (d["RgCorrected"] - d["True_R0"]) / d["True_R0"])
    .groupby("PhotonQDensity")
    .agg(mean_V_discrepancy=("V_disc", "mean"), std_V_discrepancy=("V_disc", "std"),
         mean_R_discrepancy=("R_disc", "mean"), std_R_discrepancy=("R_disc", "std"))
    .reset_index()
)
print("\nSummary by collected photon density:")
display(summary)


## Next steps

- Clone the repo (`!git clone https://github.com/roybeckbarkai/tenor-saxs.git`) to run the full paper-figure reproduction scripts in `scripts/` (`reproduce_fig2.py`, `reproduce_fig3.py`, `reproduce_fig5_r0_sensitivity.py`, `reproduce_fig6_psf0_sensitivity.py`, `reproduce_fig6_noise_violin.py`, `reproduce_fig7_nradii_sensitivity.py`, `reproduce_fig8_pxn_sensitivity.py`, `reproduce_calibration_vs_analytic.py`), or the full 12-case x 7-noise-level x 30-replicate benchmark (`run_benchmark.py`).
- `scripts/fixman_recovery_test.py` and `scripts/skewness_sensitivity.py` exercise two distribution families beyond the interactive panel's built-in set (Fixman's parameter-free ideal-chain conformations, and the independently-tunable-skewness ensemble `dist_type="skewed"`), at a larger scale.
- Load your own experimental 2D detector image by building `qx`/`qy` meshgrids from your beamline geometry and calling `protocol.tenor_protocol(intensity, qx, qy, phi2=...)` directly — it accepts any 2D intensity array with matching q-coordinate meshgrids.
- The `matlab/` folder contains the original MATLAB reference implementation and an Octave export harness (`run_reference_export.m`) used to validate this Python port numerically against it.